## 1. Importation des packages et données brutes

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_rows', 500)

import requests
from bs4 import BeautifulSoup
import os
import s3fs
import ast
import json

Les données de movies_metadata.csv et credits.csv sont des données trouvées sur Kaggle qui centralisent des informations diverses sur des films sortis avant juillet 2017. 

Les variables de movies_metadata.csv sont :
- adult : signification de la variable non connue
- belongs to collection : si le film appartient à une série de film, la variable renseigne les films faisant partie de cette série
- budget : budget du film
- genres : genres du film
- homepage : lien vers le site officiel du film s'il y en a un
- id et imdb_id : identifiants du film
- original_language : langue originale du film
- original_title : titre original du film
- overview : résumé du film
- popularity : popularité du film sur IMDB
- poster_path : lien vers l'affiche du film
- production_countries : pays de production du film
- production_companies : compagnies de production du film
- release_date : date de sortie du film
- revenue : recettes du film
- runtime : durée du film
- spoken_languages : langues parlées dans le film en version originale
- status : si le film est sorti, prévu, annulé, en production etc
- tagline : catchphrase du film
- title : titre anglophone du film
- video : False si le film est sorti au cinéma, True s'il est sorti directement sur Internet et qu'il n'a pas été diffusé au cinéma

Les variables de credits.csv sont :
- cast : casting du film sous forme de liste de dictionnaires
- crew : équipe du film sous forme de liste de dictionnaires

Nous souhaitons à partir de ces données prédire la note de nouveaux films, voir quelles sont les variables les plus décisives pour prédire si un film sera ien reçu par le public et ainsi remarquer (ou non) la prévisibilité du succès d'un film.

Nous pourrons pondérer l'erreur de prévision avec la variable vote_count et faire de la classification non supervisée dans les stats descriptives

In [3]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_KEY_S3 = '/movies_metadata.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_movies = pd.read_csv(file_in,sep=',', header=0)

/tmp/ipykernel_45896/2309787388.py:9: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data_movies = pd.read_csv(file_in,sep=',', header=0)


In [4]:
FILE_KEY_S3 = '/credits.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_credits = pd.read_csv(file_in,sep=',', header=0)

# 2. Retraitements

Après première exploration des données movies_metadata, nous avons décidé d'enlever les variables suivantes : adult, homepage, overview, popularity, poster_path, spoken_languages et tagline. En effet, la variable adult présente presque toujours la modalité False et semble avoir peu d'intérêt. La variable homepage renseigne le lien vers le site officiel du film s'il y en a un. La variable overview contient les résumés des films du dataframe, ce qui peut être intéressant à exploiter mais nous avons décidé de ne pas le faire. La variable popularity est la popularité du film sur IMDB au moment où les données ont été extraites, c'est donc une variable qui n'est pas statique et qui est calculée directement par IMDB d'une façon que nous ignorons donc nous ne souhaitons pas la prendre en compte. La variable poster_path indique le lien vers l'affiche du film, nous n'en avons pas besoin. La variable spoken_languages indique les langues parlées durant le film en version originale, nous considérons que cette variable est redondante par rapport à la variable original_language. Enfin la variable tagline indique la catchphrase du film, ce qui est à nos yeux peu utile également.

Nous allons retraiter certaines variables. Par exemple, la variable belongs_to_collection sera transformée en booléen (1 si le film correspond à une série de films, 0 sinon) à laquelle nous ajouterons une variable avec le nombre de films précédents de la série ainsi que la note du film précédent. La variable genres sera décomposée en plusieurs variables genre_1, genre_2 etc. Ce genre de décomposition sera également nécessaire pour les variables production_countries et production_companies

Les variables budget et runtime présentent des valeurs manquantes, que nous allons essayer de compléter avec du web scraping.

Nous allons nous concentrer sur les films qui sont déjà sortis en salle (status = Released et video=False)

Les données du fichier credits.csv vont nous permettre d'ajouter les acteurs principaux et le réalisateur du film à notre jeu de données. Nous souhaitons ajouter des variables relatives à la popularité des acteurs et du réalisateur via du web scraping.

Nous n'allons pas prendre en compte la variable revenue dans l'étude Machine Learning car nous souhaitons être capables de prédire la note d'un film qui ne serait pas encore sorti, donc les recettes du film ne sont pas connues à l'avance.


In [28]:
data_movies_df = data_movies[data_movies['video'] == False]
data_movies_df = data_movies_df[data_movies_df['status'] == 'Released']
data_movies_df = data_movies_df[data_movies_df['vote_count'] > 0]
data_movies_df = data_movies_df.drop(columns=['adult', 'homepage', 'overview', 'popularity', 'poster_path', 'spoken_languages', 'tagline', 'status', 'video'])

In [ ]:
data_movies_df = data_movies[data_movies['video'] == False]
data_movies_df = data_movies_df[data_movies_df['status'] == 'Released']
data_movies_df = data_movies_df[data_movies_df['vote_count'] > 0]
data_movies_df = data_movies_df.drop(columns=['adult', 'homepage', 'overview', 'popularity', 'poster_path', 'spoken_languages', 'tagline', 'status', 'video'])


In [31]:
data_movies_df = data_movies_df.dropna(subset= ['release_date'])
data_movies_df = data_movies_df.dropna(subset= ['imdb_id'])
data_movies_df = data_movies_df.dropna(subset= ['original_language'])
data_movies_df= data_movies_df.drop_duplicates()
data_movies_df = data_movies_df.groupby('id', group_keys=False).apply(lambda group: group.loc[group['vote_count'] == group['vote_count'].min()]).reset_index(drop=True)

/tmp/ipykernel_45896/3262332.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_movies_df = data_movies_df.groupby('id', group_keys=False).apply(lambda group: group.loc[group['vote_count'] == group['vote_count'].min()]).reset_index(drop=True)


## 2.1. Retraitement des variables budget et runtime

La variable budget a été reconnue comme chaîne de caractères par Python, nous la convertissons donc en numeric. Pour les variables budget et runtime lorsque la valeur n'est pas connue c'est un 0 qui s'affiche, nous remplaçons donc tous les 0 par NaN.

In [32]:
# conversion de "budget" en nombre
#data_movies_df["budget"] = pd.to_numeric(data_movies_df["budget"], errors="coerce")

# budget et duree (runtime) nuls a considerer comme valeurs manquantes
data_movies_df["budget"] = data_movies_df["budget"].replace('0', np.nan)
data_movies_df["runtime"] = data_movies_df["runtime"].replace(0, np.nan)

In [33]:
df=data_movies_df

## 2.2. Retraitement de la variable belongs_to_collection

Nous souhaitons ajouter un booléen qui vaut 1 si le film fait partie d'une saga de films et 0 sinon, le nom de la saga à laquelle il appartient, le rang du film dans la saga ainsi que le nombre de films que contient la saga au total et la note du film précédent dans la saga.  

In [34]:
# belongs to collection : 
# on crée un booleen indic_collec 
df["indic_collec"]  = df["belongs_to_collection"].notna().astype("int8") # int8 → 1 octet par valeur
df["indic_collec"].value_counts()
df_collec = df[df["indic_collec"] == 1]
df_collec['belongs_to_collection'] = df_collec['belongs_to_collection'].apply(ast.literal_eval)

# on cree une fonction pour recuperer le nom de la collection s il est rempli, rien sinon
def recup_collec(x):
    if pd.notna(x):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return None
    return x

df['belongs_to_collection'] = df['belongs_to_collection'].apply(recup_collec)
df['nom_collec'] = df['belongs_to_collection'].apply(
            lambda x: x['name'] if isinstance(x, dict) and 'name' in x else None
                                                )

/tmp/ipykernel_45896/4206164378.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_collec['belongs_to_collection'] = df_collec['belongs_to_collection'].apply(ast.literal_eval)


In [35]:
# creation d'une colonne rang
# on trie par collec et date de sortie
df = df.sort_values(by=['nom_collec', 'release_date'], ascending=[True, True])
df["rang"] = np.nan # initialisation
masque = df["nom_collec"].notna() # si collec n'est pas manquant
df.loc[masque, "rang"] = df.loc[masque].groupby("nom_collec").cumcount() + 1
#df.rang.value_counts(dropna=False)

In [36]:
# on rajoute une colonne rang max
df_max_rang = df.groupby('nom_collec')['rang'].max().reset_index()
df_max_rang.rename(columns={'rang': 'max_rang'}, inplace=True)
df = df.merge(df_max_rang, on='nom_collec', how='left')

In [37]:
#  ceux qui n'ont qu'un seul opus present sont a considerer comme ne faisant pas partie d'une serie
condition = (df['rang'] == 1) & (df['max_rang'] == 1)

df.loc[condition, 'indic_collec'] = 0
df.loc[condition, 'nom_collec'] = None
df.loc[condition, 'rang'] = np.nan
df.loc[condition, 'max_rang'] = np.nan

#df['max_rang']

In [38]:
# pour les sagas, on recupere la note de l opus precedent
df_prec = df[df['rang'] >= 1]

df_prec['rang'] += 1  # rang dans df_prec correspondra à rang - 1 dans df
df_prec = df_prec.rename(columns={'vote_average': 'vote_prec'}) # colonne vote_average renommée poru etre gardee lors de la fusion
df_prec = df_prec[['rang','nom_collec','vote_prec']]


# Merge sur nom_collec et rang
df2 = df.merge(df_prec[['nom_collec', 'rang', 'vote_prec']],
                     on=['nom_collec', 'rang'],
                     how='left')

df = df2.drop(columns=['belongs_to_collection'])

/tmp/ipykernel_45896/2582065488.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prec['rang'] += 1  # rang dans df_prec correspondra à rang - 1 dans df


In [39]:
df.shape

(42028, 19)

## 2.3. Retraitement de la variable genres

In [42]:
# variable genres : a considerer comme une liste de dictionnaire
df['genres'] = df['genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
# : il peut y en avoir plusieurs : combien maximum?
max_genres = df['genres'].apply(lambda x: len(x) if isinstance(x, list) else 0).max()
print(f"Nombre maximum de dictionnaires dans 'genres' : {max_genres}")


Nombre maximum de dictionnaires dans 'genres' : 8


In [43]:
# Creation des colonnes genre1 à genre8
for i in range(8):
    col_name = f'genre{i+1}'
    df[col_name] = df['genres'].apply(
        lambda x: x[i]['name'] if isinstance(x, list) and len(x) > i and isinstance(x[i], dict) else None
    )

In [44]:
# on regarde les modalités presentes dans toutes les colonnes genre1 a genre8
colonnes_genres = [f'genre{i}' for i in range(1, 9)]
modalites = pd.concat([df[col] for col in colonnes_genres])
frequences = modalites.value_counts()

print(frequences)


Drama              19191
Comedy             12508
Thriller            7409
Romance             6414
Action              6356
Horror              4560
Crime               4124
Documentary         3375
Adventure           3356
Science Fiction     2942
Family              2642
Mystery             2393
Fantasy             2240
Animation           1836
Foreign             1528
Music               1456
History             1336
War                 1267
Western              936
TV Movie             700
Name: count, dtype: int64


## 2.4. Retraitement de la variable production_companies

In [ ]:
# variable production_companies : a considerer comme une liste de dictionnaire
df['production_companies'] = df['production_companies'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
# : il peut y en avoir plusieurs : combien maximum?
max_prod = df['production_companies'].apply(lambda x: len(x) if isinstance(x, list) else 0).max()
#print(f"Nombre maximum de dictionnaires dans 'production_companies' : {max_prod}")
# a voir : il ne sera pas pertinent de garder 26 colonnes

In [ ]:
# Creation des colonnes prod1 à prod26
for i in range(26):
    col_name = f'prod{i+1}'
    df[col_name] = df['production_companies'].apply(
        lambda x: x[i]['name'] if isinstance(x, list) and len(x) > i and isinstance(x[i], dict) else None
    )

## 2.5. Retraitement de la variable production_countries

In [ ]:
# variable production_countries : a considerer comme une liste de dictionnaire
df['production_countries'] = df['production_countries'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
# : il peut y en avoir plusieurs : combien maximum?
max_prod = df['production_countries'].apply(lambda x: len(x) if isinstance(x, list) else 0).max()
#print(f"Nombre maximum de dictionnaires dans 'production_countries' : {max_prod}")
# a voir : il ne sera pas pertinent de garder 25 colonnes

In [ ]:
# Creation des colonnes pays1 a pays25
for i in range(25):
    col_name = f'pays{i+1}'
    df[col_name] = df['production_countries'].apply(
        lambda x: x[i]['name'] if isinstance(x, list) and len(x) > i and isinstance(x[i], dict) else None
    )

In [ ]:
missing_percentage = df.isna().sum()

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

In [ ]:
df.shape

In [ ]:
data_movies_df=df

## 2.6. Web scraping budget sur Wikipédia

Nous souhaitons compléter la variable budget qui présente beaucoup de valeurs manquantes, ainsi qu'ajouter des informations sur les acteurs et réalisateurs.

On ajoute au dataframe les différents url wikipédia possibles pour un film (selon le nom du film, il faut parfois ajouter film ou film + année de sortie à l'url wikipédia pour tomber sur la bonne page wiki)

In [ ]:
url_wikipedia_fr = "https://fr.wikipedia.org/wiki/"
url_wikipedia_en = "https://en.wikipedia.org/wiki/"
data_movies_df['url'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_")
data_movies_df['url_film'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_") + "_(film)"
data_movies_df['release_year'] = data_movies_df.release_date.str[:4]
data_movies_df['url_film_date'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_") + "_(film,_" + data_movies_df.release_year + ")"
data_movies_df['id'] = pd.to_numeric(data_movies_df['id'])


On joint les dataframes movies et credits pour ajouter le casting et l'équipe du film.

In [ ]:
data_movies_credits = data_movies_df.merge(data_credits.drop_duplicates(), left_on='id', right_on='id', how='left')
data_movies_credits


In [ ]:
data_movies_credits.shape

On retraite les colonnes cast et crew pour que Python les reconnaissent en tant que liste de dictionnaires.

In [ ]:
data_movies_credits = data_movies_credits.dropna(subset= ['cast'])

data_movies_credits['cast'] = data_movies_credits['cast'].apply(ast.literal_eval)
data_movies_credits['crew'] = data_movies_credits['crew'].apply(ast.literal_eval)

In [ ]:
data_movies_credits["nb_acteurs"] = data_movies_credits["cast"].apply(len)
films_par_nombre_acteurs = data_movies_credits["nb_acteurs"].value_counts().sort_index()
films_par_nombre_acteurs

In [ ]:
for i in range(36):
    col_name = f'acteur{i+1}'
    data_movies_credits[col_name] = data_movies_credits['cast'].apply(
        lambda x: x[i]['name'] if isinstance(x, list) and len(x) > i and isinstance(x[i], dict) else None
    )

On ajoute les colonnes correspondant aux 4 acteurs principaux du film et une colonne pour le réalisateur du film.

In [ ]:

data_movies_credits['realisateur'] = data_movies_credits['crew'].apply(
    lambda lst: lst['job' == 'Director']['name'] if isinstance(lst, list) and len(lst) > 0 else None
)


data_movies_credits = data_movies_credits.drop(columns=['cast', 'crew', 'genres', 'production_companies', 'production_countries'])


In [ ]:
data_movies_credits = data_movies_credits.drop_duplicates()
indices_a_exclure = [17411, 23836, 35546, 39616]
data_movies_credits = data_movies_credits.drop(indices_a_exclure)

In [ ]:
data_movies_credits.shape

In [ ]:
data_movies_credits[data_movies_credits["budget"].isna()].shape

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
}

Fonction pour trouver le budget d'un film avec l'url wikipédia

In [ ]:
def extraire_budget_depuis_wikipedia(url):
    try:
        # Charger la page
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parser le HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # Trouver l'infobox (il peut y avoir plusieurs classes, mais 'infobox' est souvent commun)
        infobox = soup.find('table', class_='infobox')

        if infobox is None:
            return None  # Pas d'infobox trouvée

        # Chercher les lignes de l'infobox
        rows = infobox.find_all('tr')

        for row in rows:
            header = row.find('th')
            if header and 'budget' in header.get_text(strip=True).lower():
                # Trouver la cellule contenant la valeur
                value_cell = row.find('td')
                if value_cell:
                    return value_cell.get_text(separator=" ", strip=True)

        return None  # Pas de ligne contenant "budget"

    except Exception as e:
        print(f"Erreur lors du traitement de {url}: {e}")
        return None


In [ ]:
data['budget_url'] = data['url'].apply(extraire_budget_depuis_wikipedia)
data['budget_url_film'] = data['url_film'].apply(extraire_budget_depuis_wikipedia)
data['budget_url_film_date'] = data['url_film_date'].apply(extraire_budget_depuis_wikipedia)

# Ton DataFrame à sauvegarder
# Exemple : df = pd.DataFrame({'col1': [1, 2], 'col2': ['a', 'b']})
BUCKET = 'mlepennec-ensae'

FILE_OUT_S3 = '/movies_budget.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3
with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    data.to_csv(f_out, index=False)

In [ ]:
FILE_KEY_S3 = '/movies_budget.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_budget = pd.read_csv(file_in,sep=',', header=0)

In [ ]:
df=data_movies_credits.merge(data_budget[["id", "budget_url", "budget_url_film", "budget_url_film_date"]].drop_duplicates(),how='left', left_on='id', right_on='id')


In [ ]:
df.shape

## 2.7. Retraitement de la variable budget

In [ ]:

# Création de la colonne 'budget_final'
df['budget_final'] = (
    df['budget']
    .fillna(df["budget_url"])
    .fillna(df['budget_url_film'])
    .fillna(df['budget_url_film_date'])
)


In [ ]:
FILE_OUT_S3 = '/movies_credits_budget.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3
with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    df.to_csv(f_out, index=False)

In [ ]:
df[df["budget_final"].isna()].shape

In [ ]:
import re
import pandas as pd

def nettoyer_budget(budget):
    if pd.isna(budget):
        return budget  # Laisse les NaN inchangés
    # Convertit en chaîne
    budget_str = str(budget)
    # Supprime la partie entre crochets (avec crochets et espaces autour)
    budget_sans_crochets = re.sub(r'\s*\[\s*.*?\s*\]', '', budget_str)
    # Supprime les virgules
    budget_sans_virgules = budget_sans_crochets.replace(',', '')
    # Supprime les espaces résiduels
    return budget_sans_virgules.strip()

df['budget_nettoye'] = df['budget_final'].apply(nettoyer_budget)


In [ ]:
df['budget_nettoye'].value_counts().head(50)

In [ ]:
def convertir_budget(chaine):
    if not isinstance(chaine, str):
        return None
    
    chaine = chaine.strip()

    # Cas 1 : $x million (avec x décimal ou entier)
    match_million = re.match(r'^\$([\d\.]+)\s+million$', chaine, re.IGNORECASE)
    if match_million:
        x = float(match_million.group(1))
        return str(int(x * 1_000_000))

    # Cas 2 : $x (valeur directe, sans "million")
    match_direct = re.match(r'^\$([\d,\.]+)$', chaine)
    if match_direct:
        x_str = match_direct.group(1).replace(',', '')  # supprime les virgules si présentes
        try:
            x = float(x_str)
            return str(int(x)) if x.is_integer() else str(x)
        except ValueError:
            return None

    # Sinon : format non reconnu
    return None

df['budget_converti'] = df['budget_nettoye'].apply(convertir_budget)
df['budget_converti'] = df['budget_converti'].fillna(df["budget_nettoye"])

In [ ]:
df['budget_converti'][24000]

In [ ]:

def convertir_crore(chaine):
    if not isinstance(chaine, str):
        return None
    
    chaine = chaine.strip()

    # ₹x crore
    match_crore = re.match(r'^₹\s*([\d\.]+)\s*crore$', chaine, re.IGNORECASE)
    if match_crore:
        x = float(match_crore.group(1))
        return str(x * 113000)

    # ₹x million
    match_million = re.match(r'^₹\s*([\d\.]+)\s*million$', chaine, re.IGNORECASE)
    if match_million:
        x = float(match_million.group(1))
        return str(x * 0.011)

    # Aucun format reconnu
    return None

df['budget_converti2'] = df['budget_converti'].apply(convertir_crore)
df['budget_converti2'] = df['budget_converti2'].fillna(df["budget_converti"])

In [ ]:
import pandas as pd

def est_chaine_numerique(val):
    if not isinstance(val, str):
        return False
    try:
        float(val)
        return True
    except ValueError:
        return False

# Filtrer les lignes où 'budget' N'est PAS une chaîne numérique
df_avec_budget=df[~df['budget_final'].isna()]
df_filtré = df_avec_budget[~df_avec_budget['budget_converti2'].apply(est_chaine_numerique)]
df_filtré

In [ ]:
df['release_year'].value_counts()

In [ ]:
df_nan_budget = df[df['release_year']<1960]

# Compter le nombre de lignes par 'release_year'
count_by_year = df_nan_budget.groupby('budget_final').size()
count_by_year

In [ ]:
df[df['budget_final'].isna() & (df['release_year'] < 1960)]


## 2.8. Web scraping récompenses des acteurs et réalisateurs

### 2.8.1. Web scraping acteurs

In [ ]:
colonnes_acteurs = [col for col in df.columns if col.startswith('acteur')]
tous_les_acteurs = df[colonnes_acteurs].values.flatten()
acteurs_uniques = pd.Series(tous_les_acteurs).dropna().unique().tolist()
acteurs_uniques = pd.DataFrame(acteurs_uniques, columns=['acteur'])

acteurs_uniques

In [ ]:
acteurs_uniques['url_acteur']= url_wikipedia_en + acteurs_uniques.acteur.str.replace(" ", "_")
acteurs_uniques

In [ ]:
FILE_OUT_S3 = '/acteurs_df.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3
with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    acteurs_uniques.to_csv(f_out, index=False)

In [ ]:
AWARD_KEYWORDS = [
    r'Academy Awards', r'Oscar', 'Oscars', r'Golden Globe Awards', 'Golden Globe', r'BAFTA', r'BAFTA Awards', 'BAFTAs',
    r'Filmfare', r'National Film Award', r'Dadasaheb Phalke', r'César', 'Césars',
    r'Palme d\'Or', r'Palmes d\'Or',  r'Golden Lion', r'Lion d\'Or', r'Ours d\'Or', r'Tony',
    r'Emmy Awards', r'Grammy Awards', r'Screen Actors Guild Awards', r'IIFA', r'Goya', r'David di Donatello'
]


In [52]:
import requests
from bs4 import BeautifulSoup

def get_actor_awards_structured(url):
    """
    Récupère les récompenses et nominations d'un acteur depuis sa page Wikipédia anglaise,
    et les retourne sous forme structurée.
    
    Args:
        url (str): URL de la page Wikipédia de l'acteur (en anglais)
    
    Returns:
        list: Liste de dictionnaires contenant les récompenses et nominations
    """
    headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
}
    print(url)
    try:
        response = requests.get(url, headers=headers)

    except Exception:
        return [404]
    try:
        soup = BeautifulSoup(response.text, 'html.parser')
    
    # Chercher toutes les sections qui concernent Awards / Nominations
        awards_sections = []
        for header in soup.find_all(['h2','h3']):
            if 'award' in header.get_text().strip().lower() or 'accolade' in header.get_text().strip().lower():
                awards_sections.append(header)
    
        if not awards_sections:
            return []

        awards_list = []

        for section in awards_sections:
            sibling=section.find_previous().find_next_sibling()
    
            pattern="list of awards and nominations received by"
            if pattern in section.find_previous().find_next_sibling().get_text().lower():

                new_url= "https://en.wikipedia.org/wiki/" +'List_of_awards_and_nominations_received_by_'+url[len(url_wikipedia_en):]

                response = requests.get(new_url, headers=headers)
                soup = BeautifulSoup(response.text, 'html.parser')
        
                awards_sections = []
                for header in soup.find_all(['h2']):
                    if header.get_text() in AWARD_KEYWORDS:
                        awards_sections.append(header)
        
                awards_list=["Page des awards"]
                for section in awards_sections:
                    sibling=section.find_previous().find_next_sibling()
                    if sibling.name == 'table' and 'wikitable' in sibling.get('class', []):
                        for row in sibling.find_all('tr')[1:]:  # ignorer l'entête
                            cols = [c.get_text(separator=" ", strip=True) for c in row.find_all(['td', 'th'])]
                            if len(cols) >= 3:
                                awards_list.append({
                                "year": cols[0],
                                "award": cols[1],
                                "result": cols[2],
                                "project": cols[3] if len(cols) > 3 else None
                            })
                return awards_list
            else:
                if sibling.name == 'table' and 'wikitable' in sibling.get('class', []):
                    for row in sibling.find_all('tr')[1:]:  # ignorer l'entête
                        cols = [c.get_text(separator=" ", strip=True) for c in row.find_all(['td', 'th'])]
                        if len(cols) >= 3:
                            awards_list.append({
                                "year": cols[0],
                                "award": cols[1],
                                "result": cols[2],
                                "project": cols[3] if len(cols) > 3 else None
                            })


                elif sibling.name == 'ul':
                        for li in sibling.find_all('li'):
                            text = li.get_text(separator=" ", strip=True)
                    # On peut essayer d'extraire l'année si elle est en début de texte
                            parts = text.split("–")  # souvent "2020 – Award Name"
                            if len(parts) == 2:
                                year, award_name = parts
                            else:
                                year, award_name = None, text
                            awards_list.append({
                            "year": year.strip() if year else None,
                            "award": award_name.strip(),
                            "result": None,  # pas toujours disponible ici
                            "project": None  # pas toujours disponible ici
                        })

        return awards_list
    except Exception:
        return ["Erreur"]




In [ ]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_OUT_S3 = '/acteurs_df.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3
with fs.open(FILE_OUT_PATH, mode='rb') as f_out:
    acteurs_df=pd.read_csv(f_out)

In [ ]:
acteurs_df_4 = acteurs_df.iloc[135001:len(acteurs_df)]
acteurs_df_4

In [ ]:
url_wikipedia_en = "https://en.wikipedia.org/wiki/"
acteurs_df_4['awards_info'] = acteurs_df_4['url_acteur'].apply(get_actor_awards_structured)
FILE_OUT_S3 = '/acteurs_df_4.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3
with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    acteurs_df_4.to_csv(f_out, index=False)

In [39]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_KEY_S3 = '/acteurs_df_1.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    acteurs_df_1 = pd.read_csv(file_in,sep=',', header=0)

In [40]:
FILE_KEY_S3 = '/acteurs_df_2.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    acteurs_df_2 = pd.read_csv(file_in,sep=',', header=0)

In [41]:
FILE_KEY_S3 = '/acteurs_df_3.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    acteurs_df_3 = pd.read_csv(file_in,sep=',', header=0)

In [42]:
FILE_KEY_S3 = '/acteurs_df_4.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    acteurs_df_4 = pd.read_csv(file_in,sep=',', header=0)

In [43]:
acteurs_df_concat = pd.concat([acteurs_df_1, acteurs_df_2, acteurs_df_3, acteurs_df_4], axis=0)

In [ ]:
acteurs_df_concat=acteurs_df_concat.reset_index(drop=True)
acteurs_df_concat

,acteur,url_acteur,awards_info
0,Gerard Butler,https://en.wikipedia.org/wiki/Gerard_Butler,"[{'year': '2004', 'award': 'Dear Frankie', 're..."
1,Aaron Eckhart,https://en.wikipedia.org/wiki/Aaron_Eckhart,"[{'year': None, 'award': 'Independent Spirit A..."
2,Angela Bassett,https://en.wikipedia.org/wiki/Angela_Bassett,"[{'year': '1994', 'award': 'Best Actress', 're..."
3,Morgan Freeman,https://en.wikipedia.org/wiki/Morgan_Freeman,"[{'year': '1988', 'award': 'Best Supporting Ac..."
4,Radha Mitchell,https://en.wikipedia.org/wiki/Radha_Mitchell,"[{'year': '2001', 'award': 'Fangoria Chainsaw ..."
5,Rick Yune,https://en.wikipedia.org/wiki/Rick_Yune,[]
6,Dylan McDermott,https://en.wikipedia.org/wiki/Dylan_McDermott,"[{'year': '1998', 'award': 'Viewers for Qualit..."
7,Finley Jacobsen,https://en.wikipedia.org/wiki/Finley_Jacobsen,[]
8,Tory Kittles,https://en.wikipedia.org/wiki/Tory_Kittles,[]
9,Shanna Forrestall,https://en.wikipedia.org/wiki/Shanna_Forrestall,[]


In [70]:
acteurs_df_1[acteurs_df_1['acteur']=='Daniel Craig']

,acteur,url_acteur,awards_info
11289,Daniel Craig,https://en.wikipedia.org/wiki/Daniel_Craig,[]


In [ ]:
acteurs_df_concat['awards_info'] = acteurs_df_concat['awards_info'].apply(ast.literal_eval)

In [ ]:
df_avec_recompenses = acteurs_df_concat[acteurs_df_concat['awards_info'].apply(lambda x: x != [])].reset_index(drop=True)
df_avec_recompenses

In [ ]:
df_avec_recompenses.iloc[3,2]

In [ ]:
url_wikipedia_en = "https://en.wikipedia.org/wiki/"
df_avec_recompenses['awards_info_2'] = df_avec_recompenses['url_acteur'].apply(get_actor_awards_structured)

In [ ]:
df_avec_recompenses

In [ ]:
df_avec_recompenses.iloc[3,3]

In [ ]:
FILE_OUT_S3 = '/acteurs_df_awards.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3
with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    df_avec_recompenses.to_csv(f_out, index=False)

In [ ]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_OUT_S3 = '/donnees_retraitees.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3


with fs.open(FILE_OUT_PATH, mode='rb') as f_out:
    donnees_retraitees=pd.read_csv(f_out)

In [ ]:
FILE_OUT_S3 = '/acteurs_df_awards.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3


with fs.open(FILE_OUT_PATH, mode='rb') as f_out:
    acteurs_df_awards=pd.read_csv(f_out)

In [ ]:
acteurs_df_awards.info()

In [ ]:
data_movies_final = donnees_retraitees.drop(['acteur1', 'acteur2', 'acteur3', 'acteur4'], axis=1).merge(data_movies_credits[['id', 'acteur1', 'acteur2', 'acteur3', 'acteur4', 'acteur5', 'acteur6', 'acteur7', 'acteur8', 'acteur9', 'acteur10']], left_on='id', right_on='id', how='left')

In [ ]:
data_movies_final

In [ ]:
data_movies_final = data_movies_final.merge(acteurs_df_awards[['acteur', 'awards_info_2']], left_on='acteur1', right_on='acteur', how='left', suffixes=('_0', '_1'))


In [ ]:
data_movies_final = data_movies_final.merge(acteurs_df_awards[['acteur', 'awards_info_2']], left_on='acteur2', right_on='acteur', how='left', suffixes=('_1', '_2'))

In [ ]:
data_movies_final = data_movies_final.merge(acteurs_df_awards[['acteur', 'awards_info_2']], left_on='acteur10', right_on='acteur', how='left', suffixes=('_9', '_10'))

In [ ]:
data_movies_final.info()

### 2.8.2. Web scraping réalisateurs

In [ ]:
colonnes_real = [col for col in data_movies_final.columns if col.startswith('realisateur')]
tous_les_reals = data_movies_final[colonnes_real].values.flatten()
reals_uniques = pd.Series(tous_les_reals).dropna().unique().tolist()
reals_uniques = pd.DataFrame(reals_uniques, columns=['realisateur'])

reals_uniques

In [ ]:
reals_uniques['url_realisateur']= url_wikipedia_en + reals_uniques.realisateur.str.replace(" ", "_")
reals_uniques

In [ ]:
url_wikipedia_en = "https://en.wikipedia.org/wiki/"
reals_uniques['awards_info'] = reals_uniques['url_realisateur'].apply(get_actor_awards_structured)
FILE_OUT_S3 = '/real_awards.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3


In [ ]:
with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    reals_uniques.to_csv(f_out, index=False)

In [ ]:
data_movies_final = data_movies_final.merge(reals_uniques[['realisateur', 'awards_info']], left_on='realisateur', right_on='realisateur', how='left')

In [ ]:
data_movies_final[['title','release_year', 'awards_info_2_1']]

## 2.9. Retraitements récompenses des acteurs et réalisateurs

In [2]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_OUT_S3 = '/data_movies_final.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3
with fs.open(FILE_OUT_PATH, mode='rb') as f_out:
    data_movies_final=pd.read_csv(f_out)

In [19]:
data_movies_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42028 entries, 0 to 42027
Data columns (total 76 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   budget                  8763 non-null   float64
 1   id                      42028 non-null  int64  
 2   imdb_id                 42028 non-null  object 
 3   original_language       42028 non-null  object 
 4   original_title          42028 non-null  object 
 5   release_date            42028 non-null  object 
 6   revenue                 42028 non-null  float64
 7   runtime                 40726 non-null  float64
 8   title                   42028 non-null  object 
 9   vote_average            42028 non-null  float64
 10  vote_count              42028 non-null  float64
 11  indic_collec            42028 non-null  int64  
 12  nom_collec              4015 non-null   object 
 13  rang                    4015 non-null   float64
 14  max_rang                4015 non-null 

In [3]:
cols = [col for col in data_movies_final.columns if 'acteur_' in col or 'awards_info' in col]+['id', 'release_year']
df_selected = data_movies_final[cols]
df_selected

,acteur_1,awards_info_2_1,acteur_2,awards_info_2_2,acteur_3,awards_info_2_3,acteur_4,awards_info_2_4,acteur_5,awards_info_2_5,...,awards_info_2_7,acteur_8,awards_info_2_8,acteur_9,awards_info_2_9,acteur_10,awards_info_2_10,awards_info,id,release_year
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],315946,1874
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],194079,1878
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],426903,1883
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],159897,1887
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],96882,1888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42023,NaN,NaN,Anton Yelchin,"[{'year': '2002', 'award': 'Phoenix Film Criti...",NaN,NaN,Françoise Lebrun,"[{'year': '2023', 'award': 'Lumière Award for ...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],374471,2017
42024,Timothée Chalamet,"['Page des awards', {'year': '2018', 'award': ...",Armie Hammer,"[{'year': '2010', 'award': 'Alliance of Women ...",NaN,NaN,Amira Casar,"[{'year': '1998', 'award': 'Nomination: César ...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,André Aciman,"[{'year': None, 'award': '1995 Whiting Award',...","[{'year': '1999', 'award': 'Venice Film Festiv...",398818,2017
42025,NaN,NaN,Peter Bogdanovich,"[{'year': '1972', 'award': 'Academy Awards', '...",Karyn Kusama,"[{'year': '2000', 'award': 'Girlfight', 'resul...",NaN,NaN,NaN,NaN,...,"[{'year': '2005', 'award': 'Fantasporto Awards...",NaN,NaN,NaN,NaN,NaN,NaN,[],359749,2017
42026,Jon Bernthal,"[{'year': '2011', 'award': 'Scream Awards', 'r...",Christopher Abbott,"[{'year': '2011', 'award': 'Gotham Award', 're...",NaN,NaN,Imogen Poots,"[{'year': '2007', 'award': 'British Independen...",Odessa Young,"[{'year': '2016', 'award': 'AACTA Awards', 're...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],413992,2017


In [4]:
def safe_literal_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    else:
        return []


In [5]:
for col in df_selected.columns :
    if 'awards_info' in col:
        df_selected[col] = df_selected[col].apply(safe_literal_eval)


/tmp/ipykernel_87405/3456380350.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[col] = df_selected[col].apply(safe_literal_eval)


In [6]:
df_selected_1 =df_selected[df_selected['awards_info_2_1'].apply(lambda x: x != [])].reset_index(drop=True)
df_selected_2 =df_selected[df_selected['awards_info_2_2'].apply(lambda x: x != [])].reset_index(drop=True)
df_selected_3 =df_selected[df_selected['awards_info_2_3'].apply(lambda x: x != [])].reset_index(drop=True)
df_selected_4 =df_selected[df_selected['awards_info_2_4'].apply(lambda x: x != [])].reset_index(drop=True)
df_selected_5 =df_selected[df_selected['awards_info_2_5'].apply(lambda x: x != [])].reset_index(drop=True)
df_selected_6 =df_selected[df_selected['awards_info_2_6'].apply(lambda x: x != [])].reset_index(drop=True)
df_selected_7 =df_selected[df_selected['awards_info_2_7'].apply(lambda x: x != [])].reset_index(drop=True)
df_selected_8 =df_selected[df_selected['awards_info_2_8'].apply(lambda x: x != [])].reset_index(drop=True)
df_selected_9 =df_selected[df_selected['awards_info_2_9'].apply(lambda x: x != [])].reset_index(drop=True)
df_selected_10 =df_selected[df_selected['awards_info_2_10'].apply(lambda x: x != [])].reset_index(drop=True)

df_selected_real =df_selected[df_selected['awards_info'].apply(lambda x: x != [])].reset_index(drop=True)

In [7]:
df_selected_1 =df_selected_1[['id', 'release_year', 'acteur_1', 'awards_info_2_1']]
df_selected_2 =df_selected_2[['id', 'release_year', 'acteur_2', 'awards_info_2_2']]
df_selected_3 =df_selected_3[['id', 'release_year', 'acteur_3', 'awards_info_2_3']]
df_selected_4 =df_selected_4[['id', 'release_year', 'acteur_4', 'awards_info_2_4']]
df_selected_5 =df_selected_5[['id', 'release_year', 'acteur_5', 'awards_info_2_5']]
df_selected_6 =df_selected_6[['id', 'release_year', 'acteur_6', 'awards_info_2_6']]
df_selected_7 =df_selected_7[['id', 'release_year', 'acteur_7', 'awards_info_2_7']]
df_selected_8 =df_selected_8[['id', 'release_year', 'acteur_8', 'awards_info_2_8']]
df_selected_9 =df_selected_9[['id', 'release_year', 'acteur_9', 'awards_info_2_9']]
df_selected_10 =df_selected_10[['id', 'release_year', 'acteur_10', 'awards_info_2_10']]

df_selected_real =df_selected_real[['id', 'release_year', 'awards_info']]

In [8]:
df_selected_1_big_actors = df_selected_1[df_selected_1['awards_info_2_1'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]
df_selected_2_big_actors = df_selected_2[df_selected_2['awards_info_2_2'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]
df_selected_3_big_actors = df_selected_3[df_selected_3['awards_info_2_3'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]
df_selected_4_big_actors = df_selected_4[df_selected_4['awards_info_2_4'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]
df_selected_5_big_actors = df_selected_5[df_selected_5['awards_info_2_5'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]
df_selected_6_big_actors = df_selected_6[df_selected_6['awards_info_2_6'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]
df_selected_7_big_actors = df_selected_7[df_selected_7['awards_info_2_7'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]
df_selected_8_big_actors = df_selected_8[df_selected_8['awards_info_2_8'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]
df_selected_9_big_actors = df_selected_9[df_selected_9['awards_info_2_9'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]
df_selected_10_big_actors = df_selected_10[df_selected_10['awards_info_2_10'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]

df_selected_real_big_actors = df_selected_real[df_selected_real['awards_info'].apply(lambda x: len(x) > 0 and x[0]== 'Page des awards')]

In [9]:
df_selected_1_big_actors['awards_info_2_1'] = df_selected_1_big_actors['awards_info_2_1'].apply(lambda x: x[1:] if len(x) > 1 else [])
df_selected_2_big_actors['awards_info_2_2'] = df_selected_2_big_actors['awards_info_2_2'].apply(lambda x: x[1:] if len(x) > 1 else [])
df_selected_3_big_actors['awards_info_2_3'] = df_selected_3_big_actors['awards_info_2_3'].apply(lambda x: x[1:] if len(x) > 1 else [])
df_selected_4_big_actors['awards_info_2_4'] = df_selected_4_big_actors['awards_info_2_4'].apply(lambda x: x[1:] if len(x) > 1 else [])
df_selected_5_big_actors['awards_info_2_5'] = df_selected_5_big_actors['awards_info_2_5'].apply(lambda x: x[1:] if len(x) > 1 else [])
df_selected_6_big_actors['awards_info_2_6'] = df_selected_6_big_actors['awards_info_2_6'].apply(lambda x: x[1:] if len(x) > 1 else [])
df_selected_7_big_actors['awards_info_2_7'] = df_selected_7_big_actors['awards_info_2_7'].apply(lambda x: x[1:] if len(x) > 1 else [])
df_selected_8_big_actors['awards_info_2_8'] = df_selected_8_big_actors['awards_info_2_8'].apply(lambda x: x[1:] if len(x) > 1 else [])
df_selected_9_big_actors['awards_info_2_9'] = df_selected_9_big_actors['awards_info_2_9'].apply(lambda x: x[1:] if len(x) > 1 else [])
df_selected_10_big_actors['awards_info_2_10'] = df_selected_10_big_actors['awards_info_2_10'].apply(lambda x: x[1:] if len(x) > 1 else [])

df_selected_real_big_actors['awards_info'] = df_selected_real_big_actors['awards_info'].apply(lambda x: x[1:] if len(x) > 1 else [])

/tmp/ipykernel_87405/2646283607.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected_1_big_actors['awards_info_2_1'] = df_selected_1_big_actors['awards_info_2_1'].apply(lambda x: x[1:] if len(x) > 1 else [])
/tmp/ipykernel_87405/2646283607.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected_2_big_actors['awards_info_2_2'] = df_selected_2_big_actors['awards_info_2_2'].apply(lambda x: x[1:] if len(x) > 1 else [])
/tmp/ipykernel_87405/2646283607.py:3: SettingWithCopyWarning: 
A value is 

In [10]:
import re
def fix_awards_years(awards_list):
    previous_year = None
    for award in awards_list:
        year = award.get('year')
        if isinstance(year, str) and re.fullmatch(r'\d{4}', year):
            previous_year = year
        else:
            award['year'] = previous_year
    return awards_list

In [11]:
df_selected_1_big_actors['awards_info_2_1'] = df_selected_1_big_actors['awards_info_2_1'].apply(fix_awards_years)
df_selected_1_big_actors['awards_info_2_1'] = df_selected_1_big_actors['awards_info_2_1'].apply(fix_awards_years)
df_selected_2_big_actors['awards_info_2_2'] = df_selected_2_big_actors['awards_info_2_2'].apply(fix_awards_years)
df_selected_3_big_actors['awards_info_2_3'] = df_selected_3_big_actors['awards_info_2_3'].apply(fix_awards_years)
df_selected_4_big_actors['awards_info_2_4'] = df_selected_4_big_actors['awards_info_2_4'].apply(fix_awards_years)
df_selected_5_big_actors['awards_info_2_5'] = df_selected_5_big_actors['awards_info_2_5'].apply(fix_awards_years)
df_selected_6_big_actors['awards_info_2_6'] = df_selected_6_big_actors['awards_info_2_6'].apply(fix_awards_years)
df_selected_7_big_actors['awards_info_2_7'] = df_selected_7_big_actors['awards_info_2_7'].apply(fix_awards_years)
df_selected_8_big_actors['awards_info_2_8'] = df_selected_8_big_actors['awards_info_2_8'].apply(fix_awards_years)
df_selected_9_big_actors['awards_info_2_9'] = df_selected_9_big_actors['awards_info_2_9'].apply(fix_awards_years)
df_selected_10_big_actors['awards_info_2_10'] = df_selected_10_big_actors['awards_info_2_10'].apply(fix_awards_years)

df_selected_real_big_actors['awards_info'] = df_selected_real_big_actors['awards_info'].apply(fix_awards_years)

/tmp/ipykernel_87405/591466028.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected_1_big_actors['awards_info_2_1'] = df_selected_1_big_actors['awards_info_2_1'].apply(fix_awards_years)
/tmp/ipykernel_87405/591466028.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected_1_big_actors['awards_info_2_1'] = df_selected_1_big_actors['awards_info_2_1'].apply(fix_awards_years)
/tmp/ipykernel_87405/591466028.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

In [12]:
def count_awards_before_film(awards_list, film_year):
    count = 0
    for award in awards_list:
        year_str = award.get('year')
        # Vérifie que year_str est une chaîne de 4 chiffres
        if isinstance(year_str, str) and re.fullmatch(r'\d{4}', year_str):
            year_int = int(year_str)
            if year_int < film_year:
                count += 1
    return count

In [13]:
df_selected_1_big_actors['big_awards_before_film_1'] = df_selected_1_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info_2_1'], row['release_year']),
    axis=1
)
df_selected_1_big_actors['small_awards_before_film_1'] = 0

df_selected_2_big_actors['big_awards_before_film_2'] = df_selected_2_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info_2_2'], row['release_year']),
    axis=1
)
df_selected_2_big_actors['small_awards_before_film_2'] = 0

df_selected_3_big_actors['big_awards_before_film_3'] = df_selected_3_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info_2_3'], row['release_year']),
    axis=1
)
df_selected_3_big_actors['small_awards_before_film_3'] = 0

df_selected_4_big_actors['big_awards_before_film_4'] = df_selected_4_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info_2_4'], row['release_year']),
    axis=1
)
df_selected_4_big_actors['small_awards_before_film_4'] = 0

df_selected_5_big_actors['big_awards_before_film_5'] = df_selected_5_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info_2_5'], row['release_year']),
    axis=1
)
df_selected_5_big_actors['small_awards_before_film_5'] = 0

df_selected_6_big_actors['big_awards_before_film_6'] = df_selected_6_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info_2_6'], row['release_year']),
    axis=1
)
df_selected_6_big_actors['small_awards_before_film_6'] = 0

df_selected_7_big_actors['big_awards_before_film_7'] = df_selected_7_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info_2_7'], row['release_year']),
    axis=1
)
df_selected_7_big_actors['small_awards_before_film_7'] = 0

df_selected_8_big_actors['big_awards_before_film_8'] = df_selected_8_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info_2_8'], row['release_year']),
    axis=1
)
df_selected_8_big_actors['small_awards_before_film_8'] = 0

df_selected_9_big_actors['big_awards_before_film_9'] = df_selected_9_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info_2_9'], row['release_year']),
    axis=1
)
df_selected_9_big_actors['small_awards_before_film_9'] = 0

df_selected_10_big_actors['big_awards_before_film_10'] = df_selected_10_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info_2_10'], row['release_year']),
    axis=1
)
df_selected_10_big_actors['small_awards_before_film_10'] = 0

df_selected_real_big_actors['big_awards_before_film'] = df_selected_real_big_actors.apply(
    lambda row: count_awards_before_film(row['awards_info'], row['release_year']),
    axis=1
)
df_selected_real_big_actors['small_awards_before_film'] = 0

/tmp/ipykernel_87405/413394296.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected_1_big_actors['big_awards_before_film_1'] = df_selected_1_big_actors.apply(
/tmp/ipykernel_87405/413394296.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected_1_big_actors['small_awards_before_film_1'] = 0
/tmp/ipykernel_87405/413394296.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

In [14]:
df_selected_1_small_actors = df_selected_1[df_selected_1['awards_info_2_1'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]
df_selected_2_small_actors = df_selected_2[df_selected_2['awards_info_2_2'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]
df_selected_3_small_actors = df_selected_3[df_selected_3['awards_info_2_3'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]
df_selected_4_small_actors = df_selected_4[df_selected_4['awards_info_2_4'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]
df_selected_5_small_actors = df_selected_5[df_selected_5['awards_info_2_5'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]
df_selected_6_small_actors = df_selected_6[df_selected_6['awards_info_2_6'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]
df_selected_7_small_actors = df_selected_7[df_selected_7['awards_info_2_7'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]
df_selected_8_small_actors = df_selected_8[df_selected_8['awards_info_2_8'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]
df_selected_9_small_actors = df_selected_9[df_selected_9['awards_info_2_9'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]
df_selected_10_small_actors = df_selected_10[df_selected_10['awards_info_2_10'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]

df_selected_real_small_actors = df_selected_real[df_selected_real['awards_info'].apply(lambda x: len(x) > 0 and x[0]!= 'Page des awards')]


In [15]:
df_selected_1_small_actors = df_selected_1_small_actors[df_selected_1_small_actors['awards_info_2_1'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]
df_selected_2_small_actors = df_selected_2_small_actors[df_selected_2_small_actors['awards_info_2_2'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]
df_selected_3_small_actors = df_selected_3_small_actors[df_selected_3_small_actors['awards_info_2_3'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]
df_selected_4_small_actors = df_selected_4_small_actors[df_selected_4_small_actors['awards_info_2_4'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]
df_selected_5_small_actors = df_selected_5_small_actors[df_selected_5_small_actors['awards_info_2_5'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]
df_selected_6_small_actors = df_selected_6_small_actors[df_selected_6_small_actors['awards_info_2_6'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]
df_selected_7_small_actors = df_selected_7_small_actors[df_selected_7_small_actors['awards_info_2_7'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]
df_selected_8_small_actors = df_selected_8_small_actors[df_selected_8_small_actors['awards_info_2_8'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]
df_selected_9_small_actors = df_selected_9_small_actors[df_selected_9_small_actors['awards_info_2_9'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]
df_selected_10_small_actors = df_selected_10_small_actors[df_selected_10_small_actors['awards_info_2_10'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]

df_selected_real_small_actors = df_selected_real_small_actors[df_selected_real_small_actors['awards_info'].apply(lambda x: isinstance(x, list) and x != ["Erreur"])]

In [16]:
def add_year_2(awards_list):
    for award in awards_list:
        year_2 = None
        for key in ['year', 'award', 'result', 'project']:
            value = award.get(key)
            if isinstance(value, str):
                match = re.search(r'\b(\d{4})\b', value)
                if match:
                    year_2 = match.group(1)
                    break
        award['year_2'] = year_2
    return awards_list



In [17]:
# Appliquer sur toute la colonne
df_selected_1_small_actors['awards_info_2_1'] = df_selected_1_small_actors['awards_info_2_1'].apply(add_year_2)
df_selected_2_small_actors['awards_info_2_2'] = df_selected_2_small_actors['awards_info_2_2'].apply(add_year_2)
df_selected_3_small_actors['awards_info_2_3'] = df_selected_3_small_actors['awards_info_2_3'].apply(add_year_2)
df_selected_4_small_actors['awards_info_2_4'] = df_selected_4_small_actors['awards_info_2_4'].apply(add_year_2)
df_selected_5_small_actors['awards_info_2_5'] = df_selected_5_small_actors['awards_info_2_5'].apply(add_year_2)
df_selected_6_small_actors['awards_info_6_1'] = df_selected_6_small_actors['awards_info_2_6'].apply(add_year_2)
df_selected_7_small_actors['awards_info_2_7'] = df_selected_7_small_actors['awards_info_2_7'].apply(add_year_2)
df_selected_8_small_actors['awards_info_2_8'] = df_selected_8_small_actors['awards_info_2_8'].apply(add_year_2)
df_selected_9_small_actors['awards_info_2_9'] = df_selected_9_small_actors['awards_info_2_9'].apply(add_year_2)
df_selected_10_small_actors['awards_info_2_10'] = df_selected_10_small_actors['awards_info_2_10'].apply(add_year_2)

df_selected_real_small_actors['awards_info'] = df_selected_real_small_actors['awards_info'].apply(add_year_2)

In [18]:
def fill_missing_years(awards_list):
    last_year = None
    for item in awards_list[1:]:
        if item.get('year_2') is None and last_year is not None:
            item['year_2'] = last_year
        elif item.get('year_2') is not None:
            last_year = item['year_2']
    return awards_list

In [19]:
df_selected_1_small_actors['awards_info_2_1'] = df_selected_1_small_actors['awards_info_2_1'].apply(fill_missing_years)
df_selected_2_small_actors['awards_info_2_2'] = df_selected_2_small_actors['awards_info_2_2'].apply(fill_missing_years)
df_selected_3_small_actors['awards_info_2_3'] = df_selected_3_small_actors['awards_info_2_3'].apply(fill_missing_years)
df_selected_4_small_actors['awards_info_2_4'] = df_selected_4_small_actors['awards_info_2_4'].apply(fill_missing_years)
df_selected_5_small_actors['awards_info_2_5'] = df_selected_5_small_actors['awards_info_2_5'].apply(fill_missing_years)
df_selected_6_small_actors['awards_info_6_1'] = df_selected_6_small_actors['awards_info_2_6'].apply(fill_missing_years)
df_selected_7_small_actors['awards_info_2_7'] = df_selected_7_small_actors['awards_info_2_7'].apply(fill_missing_years)
df_selected_8_small_actors['awards_info_2_8'] = df_selected_8_small_actors['awards_info_2_8'].apply(fill_missing_years)
df_selected_9_small_actors['awards_info_2_9'] = df_selected_9_small_actors['awards_info_2_9'].apply(fill_missing_years)
df_selected_10_small_actors['awards_info_2_10'] = df_selected_10_small_actors['awards_info_2_10'].apply(fill_missing_years)

df_selected_real_small_actors['awards_info'] = df_selected_real_small_actors['awards_info'].apply(fill_missing_years)

In [20]:
def count_big_awards_before_film_2(awards_list, film_year):
    count = 0
    for award in awards_list:
        year_str = award.get('year_2')
        # Vérifie que year_str est une chaîne de 4 chiffres
        if isinstance(year_str, str) and re.fullmatch(r'\d{4}', year_str) and (award.get('award') in AWARD_KEYWORDS or award.get('year') in AWARD_KEYWORDS or award.get('result') in AWARD_KEYWORDS):
            year_int = int(year_str)
            if year_int < film_year:
                count += 1
    return count

def count_awards_before_film_2(awards_list, film_year):
    count = 0
    for award in awards_list:
        year_str = award.get('year_2')
        # Vérifie que year_str est une chaîne de 4 chiffres
        if isinstance(year_str, str) and re.fullmatch(r'\d{4}', year_str):
            year_int = int(year_str)
            if year_int < film_year:
                count += 1
    return count

In [21]:
AWARD_KEYWORDS = [
    r'Academy Awards', r'Academy Award', r'Oscar', 'Oscars', r'Golden Globe Awards', 'Golden Globe', 'Golden Globe Award', r'BAFTA', r'BAFTA Awards', 'BAFTAs',
    r'Filmfare', r'National Film Award', r'Dadasaheb Phalke', r'César', 'Césars',
    r'Palme d\'Or', r'Palmes d\'Or',  r'Golden Lion', r'Lion d\'Or', r'Ours d\'Or', r'Tony',
    r'Emmy Awards', r'Grammy Awards', r'Screen Actors Guild Awards', r'IIFA', r'Goya', r'David di Donatello'
]


In [22]:
df_selected_1_small_actors['big_awards_before_film_1'] = df_selected_1_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info_2_1'], row['release_year']),
    axis=1)
df_selected_1_small_actors['awards_before_film_1'] = df_selected_1_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info_2_1'], row['release_year']),
    axis=1)
df_selected_1_small_actors['small_awards_before_film_1'] = df_selected_1_small_actors['awards_before_film_1']-df_selected_1_small_actors['big_awards_before_film_1']
df_selected_1_small_actors=df_selected_1_small_actors.drop(['awards_before_film_1'], axis=1)


df_selected_2_small_actors['big_awards_before_film_2'] = df_selected_2_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info_2_2'], row['release_year']),
    axis=1)
df_selected_2_small_actors['awards_before_film_2'] = df_selected_2_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info_2_2'], row['release_year']),
    axis=1)
df_selected_2_small_actors['small_awards_before_film_2'] = df_selected_2_small_actors['awards_before_film_2']-df_selected_2_small_actors['big_awards_before_film_2']
df_selected_2_small_actors=df_selected_2_small_actors.drop(['awards_before_film_2'], axis=1)


df_selected_3_small_actors['big_awards_before_film_3'] = df_selected_3_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info_2_3'], row['release_year']),
    axis=1)
df_selected_3_small_actors['awards_before_film_3'] = df_selected_3_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info_2_3'], row['release_year']),
    axis=1)
df_selected_3_small_actors['small_awards_before_film_3'] = df_selected_3_small_actors['awards_before_film_3']-df_selected_3_small_actors['big_awards_before_film_3']
df_selected_3_small_actors=df_selected_3_small_actors.drop(['awards_before_film_3'], axis=1)


df_selected_4_small_actors['big_awards_before_film_4'] = df_selected_4_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info_2_4'], row['release_year']),
    axis=1)
df_selected_4_small_actors['awards_before_film_4'] = df_selected_4_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info_2_4'], row['release_year']),
    axis=1)
df_selected_4_small_actors['small_awards_before_film_4'] = df_selected_4_small_actors['awards_before_film_4']-df_selected_4_small_actors['big_awards_before_film_4']
df_selected_4_small_actors=df_selected_4_small_actors.drop(['awards_before_film_4'], axis=1)


df_selected_5_small_actors['big_awards_before_film_5'] = df_selected_5_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info_2_5'], row['release_year']),
    axis=1)
df_selected_5_small_actors['awards_before_film_5'] = df_selected_5_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info_2_5'], row['release_year']),
    axis=1)
df_selected_5_small_actors['small_awards_before_film_5'] = df_selected_5_small_actors['awards_before_film_5']-df_selected_5_small_actors['big_awards_before_film_5']
df_selected_5_small_actors=df_selected_5_small_actors.drop(['awards_before_film_5'], axis=1)


df_selected_6_small_actors['big_awards_before_film_6'] = df_selected_6_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info_2_6'], row['release_year']),
    axis=1)
df_selected_6_small_actors['awards_before_film_6'] = df_selected_6_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info_2_6'], row['release_year']),
    axis=1)
df_selected_6_small_actors['small_awards_before_film_6'] = df_selected_6_small_actors['awards_before_film_6']-df_selected_6_small_actors['big_awards_before_film_6']
df_selected_6_small_actors=df_selected_6_small_actors.drop(['awards_before_film_6'], axis=1)


df_selected_7_small_actors['big_awards_before_film_7'] = df_selected_7_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info_2_7'], row['release_year']),
    axis=1)
df_selected_7_small_actors['awards_before_film_7'] = df_selected_7_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info_2_7'], row['release_year']),
    axis=1)
df_selected_7_small_actors['small_awards_before_film_7'] = df_selected_7_small_actors['awards_before_film_7']-df_selected_7_small_actors['big_awards_before_film_7']
df_selected_7_small_actors=df_selected_7_small_actors.drop(['awards_before_film_7'], axis=1)


df_selected_8_small_actors['big_awards_before_film_8'] = df_selected_8_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info_2_8'], row['release_year']),
    axis=1)
df_selected_8_small_actors['awards_before_film_8'] = df_selected_8_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info_2_8'], row['release_year']),
    axis=1)
df_selected_8_small_actors['small_awards_before_film_8'] = df_selected_8_small_actors['awards_before_film_8']-df_selected_8_small_actors['big_awards_before_film_8']
df_selected_8_small_actors=df_selected_8_small_actors.drop(['awards_before_film_8'], axis=1)


df_selected_9_small_actors['big_awards_before_film_9'] = df_selected_9_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info_2_9'], row['release_year']),
    axis=1)
df_selected_9_small_actors['awards_before_film_9'] = df_selected_9_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info_2_9'], row['release_year']),
    axis=1)
df_selected_9_small_actors['small_awards_before_film_9'] = df_selected_9_small_actors['awards_before_film_9']-df_selected_9_small_actors['big_awards_before_film_9']
df_selected_9_small_actors=df_selected_9_small_actors.drop(['awards_before_film_9'], axis=1)


df_selected_10_small_actors['big_awards_before_film_10'] = df_selected_10_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info_2_10'], row['release_year']),
    axis=1)
df_selected_10_small_actors['awards_before_film_10'] = df_selected_10_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info_2_10'], row['release_year']),
    axis=1)
df_selected_10_small_actors['small_awards_before_film_10'] = df_selected_10_small_actors['awards_before_film_10']-df_selected_10_small_actors['big_awards_before_film_10']
df_selected_10_small_actors=df_selected_10_small_actors.drop(['awards_before_film_10'], axis=1)


df_selected_real_small_actors['big_awards_before_film'] = df_selected_real_small_actors.apply(
    lambda row: count_big_awards_before_film_2(row['awards_info'], row['release_year']),
    axis=1)
df_selected_real_small_actors['awards_before_film'] = df_selected_real_small_actors.apply(
    lambda row: count_awards_before_film_2(row['awards_info'], row['release_year']),
    axis=1)
df_selected_real_small_actors['small_awards_before_film'] = df_selected_real_small_actors['awards_before_film']-df_selected_real_small_actors['big_awards_before_film']
df_selected_real_small_actors=df_selected_real_small_actors.drop(['awards_before_film'], axis=1)

In [23]:
df_selected_1=pd.concat([df_selected_1_small_actors, df_selected_1_big_actors], axis=0)
df_selected_2=pd.concat([df_selected_2_small_actors, df_selected_2_big_actors], axis=0)
df_selected_3=pd.concat([df_selected_3_small_actors, df_selected_3_big_actors], axis=0)
df_selected_4=pd.concat([df_selected_4_small_actors, df_selected_4_big_actors], axis=0)
df_selected_5=pd.concat([df_selected_5_small_actors, df_selected_5_big_actors], axis=0)
df_selected_6=pd.concat([df_selected_6_small_actors, df_selected_6_big_actors], axis=0)
df_selected_7=pd.concat([df_selected_7_small_actors, df_selected_7_big_actors], axis=0)
df_selected_8=pd.concat([df_selected_8_small_actors, df_selected_8_big_actors], axis=0)
df_selected_9=pd.concat([df_selected_9_small_actors, df_selected_9_big_actors], axis=0)
df_selected_10=pd.concat([df_selected_10_small_actors, df_selected_10_big_actors], axis=0)

df_selected_real=pd.concat([df_selected_real_small_actors, df_selected_real_big_actors], axis=0)

In [51]:
data_movies_final_selected=data_movies_final[
    ['id',
    'runtime',
    'vote_average', 
    'vote_count', 
    'indic_collec', 
    'rang', 
    'vote_prec', 
    'release_year', 
    'release_date',
    'budget_dollar_constant',
    'North America', 
    'Europe', 
    'Asia', 
    'Latin America', 
    'Oceania', 
    'Africa',
    'Drama',
    'ComedyFamily',
    'ThrillerCrime',
    'Romance',
    'ActionAdventure',
    'WarWestern',
    'HorrorMystery',
    'SciFiFantasy',
    'Animation',
    'Intello',
    'petite',
    'moyenne',
    'grande'
    ]
]



In [52]:
data_movies_final_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42028 entries, 0 to 42027
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      42028 non-null  int64  
 1   runtime                 40726 non-null  float64
 2   vote_average            42028 non-null  float64
 3   vote_count              42028 non-null  float64
 4   indic_collec            42028 non-null  int64  
 5   rang                    4015 non-null   float64
 6   vote_prec               2728 non-null   float64
 7   release_year            42028 non-null  int64  
 8   release_date            42028 non-null  object 
 9   budget_dollar_constant  42028 non-null  float64
 10  North America           42028 non-null  int64  
 11  Europe                  42028 non-null  int64  
 12  Asia                    42028 non-null  int64  
 13  Latin America           42028 non-null  int64  
 14  Oceania                 42028 non-null

In [53]:
data_movies_final_selected_merged = data_movies_final_selected.merge(df_selected_1[['id', 'big_awards_before_film_1', 'small_awards_before_film_1']], left_on='id', right_on='id', how='left')
data_movies_final_selected_merged = data_movies_final_selected_merged.merge(df_selected_2[['id', 'big_awards_before_film_2', 'small_awards_before_film_2']], left_on='id', right_on='id', how='left')
data_movies_final_selected_merged = data_movies_final_selected_merged.merge(df_selected_3[['id', 'big_awards_before_film_3', 'small_awards_before_film_3']], left_on='id', right_on='id', how='left')
data_movies_final_selected_merged = data_movies_final_selected_merged.merge(df_selected_4[['id', 'big_awards_before_film_4', 'small_awards_before_film_4']], left_on='id', right_on='id', how='left')
data_movies_final_selected_merged = data_movies_final_selected_merged.merge(df_selected_5[['id', 'big_awards_before_film_5', 'small_awards_before_film_5']], left_on='id', right_on='id', how='left')
data_movies_final_selected_merged = data_movies_final_selected_merged.merge(df_selected_6[['id', 'big_awards_before_film_6', 'small_awards_before_film_6']], left_on='id', right_on='id', how='left')
data_movies_final_selected_merged = data_movies_final_selected_merged.merge(df_selected_7[['id', 'big_awards_before_film_7', 'small_awards_before_film_7']], left_on='id', right_on='id', how='left')
data_movies_final_selected_merged = data_movies_final_selected_merged.merge(df_selected_8[['id', 'big_awards_before_film_8', 'small_awards_before_film_8']], left_on='id', right_on='id', how='left')
data_movies_final_selected_merged = data_movies_final_selected_merged.merge(df_selected_9[['id', 'big_awards_before_film_9', 'small_awards_before_film_9']], left_on='id', right_on='id', how='left')
data_movies_final_selected_merged = data_movies_final_selected_merged.merge(df_selected_10[['id', 'big_awards_before_film_10', 'small_awards_before_film_10']], left_on='id', right_on='id', how='left')
data_movies_final_selected_merged = data_movies_final_selected_merged.merge(df_selected_real[['id', 'big_awards_before_film', 'small_awards_before_film']], left_on='id', right_on='id', how='left')

In [54]:
data_movies_final_selected_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42028 entries, 0 to 42027
Data columns (total 51 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           42028 non-null  int64  
 1   runtime                      40726 non-null  float64
 2   vote_average                 42028 non-null  float64
 3   vote_count                   42028 non-null  float64
 4   indic_collec                 42028 non-null  int64  
 5   rang                         4015 non-null   float64
 6   vote_prec                    2728 non-null   float64
 7   release_year                 42028 non-null  int64  
 8   release_date                 42028 non-null  object 
 9   budget_dollar_constant       42028 non-null  float64
 10  North America                42028 non-null  int64  
 11  Europe                       42028 non-null  int64  
 12  Asia                         42028 non-null  int64  
 13  Latin America   

In [55]:
data_movies_final_selected_merged=data_movies_final_selected_merged.rename(columns={"budget_dollar_constant": "budget",
 "Latin America": "Latin_America", 
 "North America" : "North_America", 
 "petite" : "nb_prod_comp_S", 
 "moyenne" : "nb_prod_comp_M", 
 "grande" : "nb_prod_comp_L"})

In [56]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_OUT_S3 = '/data_movies_final_merged.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3


with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    data_movies_final_selected_merged.to_csv(f_out, index=False)

# Premiers modèles

In [3]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_OUT_S3 = '/data_movies_final_merged.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3    

with fs.open(FILE_OUT_PATH, mode='rb') as f_out:
    data_movies_final_selected_merged = pd.read_csv(f_out)

In [3]:
data_movies_final_selected_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42028 entries, 0 to 42027
Data columns (total 51 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           42028 non-null  int64  
 1   runtime                      40726 non-null  float64
 2   vote_average                 42028 non-null  float64
 3   vote_count                   42028 non-null  float64
 4   indic_collec                 42028 non-null  int64  
 5   rang                         4015 non-null   float64
 6   vote_prec                    2728 non-null   float64
 7   release_year                 42028 non-null  int64  
 8   release_date                 42028 non-null  object 
 9   budget                       42028 non-null  float64
 10  North_America                42028 non-null  int64  
 11  Europe                       42028 non-null  int64  
 12  Asia                         42028 non-null  int64  
 13  Latin_America   

In [4]:
data_movies_final_selected_merged=data_movies_final_selected_merged.drop(['id'], axis=1)

In [ ]:
data_movies_final_selected_merged['runtime'] = data_movies_final_selected_merged['runtime'].fillna(value = -1)
data_movies_final_selected_merged['rang'] = data_movies_final_selected_merged['rang'].fillna(value = -1)
data_movies_final_selected_merged['vote_prec'] = data_movies_final_selected_merged['vote_prec'].fillna(value = -1)


data_movies_final_selected_merged['big_awards_before_film_1'] = data_movies_final_selected_merged['big_awards_before_film_1'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film_1'] = data_movies_final_selected_merged['small_awards_before_film_1'].fillna(value = 0)
data_movies_final_selected_merged['big_awards_before_film_2'] = data_movies_final_selected_merged['big_awards_before_film_2'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film_2'] = data_movies_final_selected_merged['small_awards_before_film_2'].fillna(value = 0)
data_movies_final_selected_merged['big_awards_before_film_3'] = data_movies_final_selected_merged['big_awards_before_film_3'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film_3'] = data_movies_final_selected_merged['small_awards_before_film_3'].fillna(value = 0)
data_movies_final_selected_merged['big_awards_before_film_4'] = data_movies_final_selected_merged['big_awards_before_film_4'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film_4'] = data_movies_final_selected_merged['small_awards_before_film_4'].fillna(value = 0)
data_movies_final_selected_merged['big_awards_before_film_5'] = data_movies_final_selected_merged['big_awards_before_film_5'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film_5'] = data_movies_final_selected_merged['small_awards_before_film_5'].fillna(value = 0)
data_movies_final_selected_merged['big_awards_before_film_6'] = data_movies_final_selected_merged['big_awards_before_film_6'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film_6'] = data_movies_final_selected_merged['small_awards_before_film_6'].fillna(value = 0)
data_movies_final_selected_merged['big_awards_before_film_7'] = data_movies_final_selected_merged['big_awards_before_film_7'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film_7'] = data_movies_final_selected_merged['small_awards_before_film_7'].fillna(value = 0)
data_movies_final_selected_merged['big_awards_before_film_8'] = data_movies_final_selected_merged['big_awards_before_film_8'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film_8'] = data_movies_final_selected_merged['small_awards_before_film_8'].fillna(value = 0)
data_movies_final_selected_merged['big_awards_before_film_9'] = data_movies_final_selected_merged['big_awards_before_film_9'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film_9'] = data_movies_final_selected_merged['small_awards_before_film_9'].fillna(value = 0)
data_movies_final_selected_merged['big_awards_before_film_10'] = data_movies_final_selected_merged['big_awards_before_film_10'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film_10'] = data_movies_final_selected_merged['small_awards_before_film_10'].fillna(value = 0)
data_movies_final_selected_merged['big_awards_before_film'] = data_movies_final_selected_merged['big_awards_before_film'].fillna(value = 0)
data_movies_final_selected_merged['small_awards_before_film'] = data_movies_final_selected_merged['small_awards_before_film'].fillna(value = 0)

In [6]:
data_movies_final_selected_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42028 entries, 0 to 42027
Data columns (total 50 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   runtime                      42028 non-null  float64
 1   vote_average                 42028 non-null  float64
 2   vote_count                   42028 non-null  float64
 3   indic_collec                 42028 non-null  int64  
 4   rang                         42028 non-null  float64
 5   vote_prec                    42028 non-null  float64
 6   release_year                 42028 non-null  int64  
 7   release_date                 42028 non-null  object 
 8   budget                       42028 non-null  float64
 9   North_America                42028 non-null  int64  
 10  Europe                       42028 non-null  int64  
 11  Asia                         42028 non-null  int64  
 12  Latin_America                42028 non-null  int64  
 13  Oceania         

In [41]:
data_movies_final_selected_merged['small_awards_before_film_10'].value_counts()

small_awards_before_film_10
0.0     40653
1.0       361
2.0       218
3.0       165
4.0       117
5.0        94
6.0        85
7.0        67
8.0        49
9.0        36
10.0       29
11.0       26
13.0       20
14.0       16
12.0       15
19.0       14
18.0        8
17.0        8
16.0        8
20.0        7
15.0        6
22.0        4
23.0        4
21.0        4
24.0        3
28.0        3
36.0        2
29.0        2
46.0        1
57.0        1
26.0        1
27.0        1
Name: count, dtype: int64

faire train_test_split random -> problématique : prévoir la note d'un nouveau film SUR IMDB peu importe la date de sortie (films récents avec moins de votes)
refaire maisons de production après le split ---> Cécile
filtrer sur les vote_count élevés (>=5) ---> Cécile
faire attention à avoir des films avec saga dans train et test
voir si on a des meilleurs résultats pour les films avec vote count élevé (>100)

importance des variables à regarder pour voir quelles variables contribuent le plus au modèle ---> Marie

refaire algorithme backward avec WLS + incompatibilité weight avec Pipeline.fit  ---> Marie
regarder l'évolution des notes selon les années de sortie  ---> Cécile